In [1]:
# ============================================================
# AI COMMUNICATION & SPEECH ANALYZER
# CELL 1 — INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q gradio openai-whisper transformers torch sentencepiece pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 20.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 5.2 MB/s eta 0:00:00


In [2]:
# ============================================================
# AI COMMUNICATION & SPEECH ANALYZER
# CELL 2 — COMMUNICATION ANALYSIS ENGINE
# ============================================================

import re
import pandas as pd


# ------------------------------------------------------------
# FILLER WORDS
# ------------------------------------------------------------

filler_words = [
    "um",
    "uh",
    "hmm",
    "like",
    "you know",
    "actually",
    "basically",
    "literally",
    "so"
]


# ------------------------------------------------------------
# COMMUNICATION ANALYZER
# ------------------------------------------------------------

def analyze_communication(text):

    if not text or not text.strip():
        return {
            "score": 0,
            "words": 0,
            "sentences": 0,
            "fillers": 0,
            "filler_list": [],
            "style": "No Input",
            "feedback": "Please enter or record some speech."
        }

    text_clean = text.strip()

    # Word count
    words = re.findall(r"\b[\w']+\b", text_clean)
    word_count = len(words)

    # Sentence count
    sentences = re.findall(r"[.!?]+", text_clean)
    sentence_count = max(len(sentences), 1)

    # Filler word detection
    lower_text = text_clean.lower()

    detected_fillers = []

    for filler in filler_words:

        pattern = r"\b" + re.escape(filler) + r"\b"

        matches = re.findall(pattern, lower_text)

        if matches:
            detected_fillers.extend([filler] * len(matches))

    filler_count = len(detected_fillers)

    # Average sentence length
    avg_sentence_length = word_count / sentence_count

    # --------------------------------------------------------
    # COMMUNICATION SCORE
    # --------------------------------------------------------

    score = 100

    # Too many filler words
    score -= min(filler_count * 4, 24)

    # Very short response
    if word_count < 10:
        score -= 10

    # Extremely long sentences
    if avg_sentence_length > 30:
        score -= 8

    # Good sentence structure
    if sentence_count >= 2:
        score += 3

    score = max(0, min(score, 100))

    # --------------------------------------------------------
    # COMMUNICATION STYLE
    # --------------------------------------------------------

    if word_count < 15:
        style = "Brief Communication"

    elif avg_sentence_length > 30:
        style = "Detailed Communication"

    elif filler_count >= 4:
        style = "Needs More Fluency"

    else:
        style = "Clear Communication"

    # --------------------------------------------------------
    # FEEDBACK
    # --------------------------------------------------------

    suggestions = []

    if filler_count > 0:
        suggestions.append(
            "Reduce filler words such as "
            + ", ".join(sorted(set(detected_fillers)))
            + "."
        )

    if word_count < 10:
        suggestions.append(
            "Try explaining your idea with a little more detail."
        )

    if avg_sentence_length > 30:
        suggestions.append(
            "Use shorter sentences to make your communication clearer."
        )

    if filler_count == 0:
        suggestions.append(
            "Excellent! No major filler-word usage was detected."
        )

    if score >= 85:
        overall = "Excellent communication clarity."

    elif score >= 70:
        overall = "Good communication with a few areas to improve."

    elif score >= 50:
        overall = "Moderate communication clarity. Practice can help."

    else:
        overall = "More practice is recommended for clearer communication."

    suggestions.append(overall)

    return {
        "score": score,
        "words": word_count,
        "sentences": sentence_count,
        "fillers": filler_count,
        "filler_list": detected_fillers,
        "style": style,
        "feedback": "\n".join("• " + item for item in suggestions)
    }


# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------

test_text = """
Hello, my name is Priya. Um, I am currently studying computer
science and I am interested in artificial intelligence.
Basically, I want to improve my communication skills.
"""

result = analyze_communication(test_text)

print("=" * 60)
print("       AI COMMUNICATION ANALYSIS")
print("=" * 60)

print("Communication Score :", result["score"], "/ 100")
print("Word Count          :", result["words"])
print("Sentence Count      :", result["sentences"])
print("Filler Words        :", result["fillers"])
print("Communication Style :", result["style"])

print("\nImprovement Feedback:")
print(result["feedback"])

       AI COMMUNICATION ANALYSIS
Communication Score : 95 / 100
Word Count          : 27
Sentence Count      : 3
Filler Words        : 2
Communication Style : Clear Communication

Improvement Feedback:
• Reduce filler words such as basically, um.
• Excellent communication clarity.


In [3]:
# ============================================================
# AI COMMUNICATION & SPEECH ANALYZER
# CELL 3 — VOICE TO TEXT ENGINE
# ============================================================

import whisper

print("Loading speech recognition model...")

speech_model = whisper.load_model("base")

print("✓ Whisper model loaded successfully")


# ------------------------------------------------------------
# VOICE TRANSCRIPTION
# ------------------------------------------------------------

def transcribe_speech(audio):

    if audio is None:
        return ""

    try:

        result = speech_model.transcribe(
            audio,
            fp16=False
        )

        return result["text"].strip()

    except Exception as e:

        return f"Transcription Error: {str(e)}"


# ------------------------------------------------------------
# VOICE → COMMUNICATION ANALYSIS
# ------------------------------------------------------------

def analyze_voice_communication(audio):

    if audio is None:
        return (
            "",
            0,
            0,
            0,
            "No Input",
            "Please record or upload your speech."
        )

    transcript = transcribe_speech(audio)

    if not transcript:
        return (
            "",
            0,
            0,
            0,
            "No Speech Detected",
            "Please try recording your voice again."
        )

    if transcript.startswith("Transcription Error"):
        return (
            transcript,
            0,
            0,
            0,
            "Error",
            transcript
        )

    analysis = analyze_communication(transcript)

    return (
        transcript,
        analysis["score"],
        analysis["words"],
        analysis["fillers"],
        analysis["style"],
        analysis["feedback"]
    )


# ------------------------------------------------------------
# TEXT TEST
# ------------------------------------------------------------

sample = """
Hello everyone. My name is Priya and I am interested in
artificial intelligence. I enjoy developing innovative
technology projects and improving my technical skills.
"""

analysis = analyze_communication(sample)

print("\n" + "=" * 60)
print("       SPEECH ANALYSIS TEST")
print("=" * 60)

print("Score :", analysis["score"], "/ 100")
print("Words :", analysis["words"])
print("Fillers:", analysis["fillers"])
print("Style :", analysis["style"])

print("\nFeedback:")
print(analysis["feedback"])

Loading speech recognition model...


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 144MiB/s]


✓ Whisper model loaded successfully

       SPEECH ANALYSIS TEST
Score : 100 / 100
Words : 24
Fillers: 0
Style : Clear Communication

Feedback:
• Excellent! No major filler-word usage was detected.
• Excellent communication clarity.


In [ ]:
# ============================================================
# AI COMMUNICATION & SPEECH ANALYZER
# CELL 4 — PROFESSIONAL INTERACTIVE UI
# ============================================================

import gradio as gr


# ------------------------------------------------------------
# TEXT ANALYSIS WRAPPER
# ------------------------------------------------------------

def analyze_text_input(text):

    result = analyze_communication(text)

    return (
        result["score"],
        result["words"],
        result["sentences"],
        result["fillers"],
        result["style"],
        result["feedback"]
    )


# ------------------------------------------------------------
# VOICE ANALYSIS WRAPPER
# ------------------------------------------------------------

def analyze_voice_input(audio):

    return analyze_voice_communication(audio)


# ------------------------------------------------------------
# SAMPLE TEXT
# ------------------------------------------------------------

def load_sample():

    return """
Hello everyone. My name is Priya. Um, I am currently
studying computer science and I am interested in artificial
intelligence. Basically, I enjoy building innovative
technology projects and improving my communication skills.
"""


# ============================================================
# CUSTOM CSS
# ============================================================

css = """

body {
    background: #111315 !important;
}

.gradio-container {
    max-width: 1250px !important;
    margin: auto !important;
    font-family: Arial, sans-serif !important;
}

/* HERO */

.hero {
    background:
        linear-gradient(135deg, #17191c, #24282d);
    border: 1px solid #3c4147;
    border-radius: 26px;
    padding: 38px;
    margin-bottom: 22px;
    box-shadow: 0 18px 45px rgba(0,0,0,0.35);
}

.hero h1 {
    color: #f4c76a;
    font-size: 40px;
    margin-bottom: 8px;
}

.hero p {
    color: #c9cdd2;
    font-size: 16px;
    margin: 6px 0;
}

/* CARDS */

.panel {
    background: #1a1d20;
    border: 1px solid #34393f;
    border-radius: 20px;
    padding: 24px;
    box-shadow: 0 12px 30px rgba(0,0,0,0.22);
}

/* STAT BOXES */

.stat {
    background: #202429;
    border: 1px solid #393f45;
    border-radius: 16px;
}

/* BUTTONS */

.gold-btn button {
    background: #d5a94f !important;
    color: #17191c !important;
    border: none !important;
    border-radius: 11px !important;
    font-weight: 700 !important;
}

.dark-btn button {
    background: #292e33 !important;
    color: #f4c76a !important;
    border: 1px solid #444a50 !important;
    border-radius: 11px !important;
}

/* SCORE */

.score textarea,
.score input {
    font-size: 28px !important;
    font-weight: bold !important;
    text-align: center !important;
}

/* NOTICE */

.notice {
    background: #1d2024;
    border-left: 4px solid #d5a94f;
    border-radius: 12px;
    padding: 17px;
    color: #cfd3d8;
}

/* FOOTER */

.footer {
    text-align: center;
    color: #777e86;
    padding: 22px;
    font-size: 13px;
}

"""


# ============================================================
# GRADIO APP
# ============================================================

with gr.Blocks(
    title="VocalIQ — Communication Analyzer",
    css=css,
    theme=gr.themes.Base(
        primary_hue="amber",
        neutral_hue="slate"
    )
) as demo:

    # --------------------------------------------------------
    # HEADER
    # --------------------------------------------------------

    gr.HTML("""
    <div class="hero">
        <h1>🎙️ VocalIQ</h1>

        <p>
            <b>AI Communication & Speech Analyzer</b>
        </p>

        <p>
            Analyze your communication clarity, detect filler
            words, measure speech quality and receive
            personalized improvement feedback.
        </p>
    </div>
    """)

    # --------------------------------------------------------
    # INPUT SECTION
    # --------------------------------------------------------

    with gr.Row():

        # TEXT PANEL
        with gr.Column(elem_classes="panel"):

            gr.Markdown("### ✍️ Text Analysis")

            text_input = gr.Textbox(
                label="Communication Sample",
                placeholder=(
                    "Type your speech, interview answer, "
                    "presentation or introduction here..."
                ),
                lines=9
            )

            with gr.Row():

                text_analyze = gr.Button(
                    "⚡ Analyze Text",
                    elem_classes="gold-btn"
                )

                sample_btn = gr.Button(
                    "📄 Load Sample",
                    elem_classes="dark-btn"
                )

        # VOICE PANEL
        with gr.Column(elem_classes="panel"):

            gr.Markdown("### 🎤 Voice Analysis")

            gr.Markdown(
                "Record your speech or upload an audio file."
            )

            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="Voice Input"
            )

            voice_analyze = gr.Button(
                "🎧 Analyze Voice",
                elem_classes="gold-btn"
            )

    # --------------------------------------------------------
    # VOICE TRANSCRIPT
    # --------------------------------------------------------

    with gr.Group(elem_classes="panel"):

        gr.Markdown("### 📝 Speech Transcription")

        transcript = gr.Textbox(
            label="Whisper Transcription",
            lines=5,
            interactive=False
        )

    # --------------------------------------------------------
    # ANALYTICS
    # --------------------------------------------------------

    gr.Markdown("## 📊 Communication Analytics")

    with gr.Row():

        score = gr.Number(
            label="🏆 Communication Score / 100",
            value=0,
            interactive=False,
            elem_classes=["stat", "score"]
        )

        words = gr.Number(
            label="🔤 Word Count",
            value=0,
            interactive=False,
            elem_classes="stat"
        )

        sentences = gr.Number(
            label="📑 Sentences",
            value=0,
            interactive=False,
            elem_classes="stat"
        )

        fillers = gr.Number(
            label="⚠️ Filler Words",
            value=0,
            interactive=False,
            elem_classes="stat"
        )

    # --------------------------------------------------------
    # STYLE + FEEDBACK
    # --------------------------------------------------------

    with gr.Row():

        with gr.Column(elem_classes="panel"):

            communication_style = gr.Textbox(
                label="🎯 Communication Style",
                interactive=False
            )

        with gr.Column(elem_classes="panel"):

            feedback = gr.Textbox(
                label="💡 AI Improvement Feedback",
                lines=6,
                interactive=False
            )

    # --------------------------------------------------------
    # INFORMATION
    # --------------------------------------------------------

    gr.HTML("""
    <div class="notice">
        <b>💡 How VocalIQ works</b><br><br>

        1. Enter text or record your voice.<br>
        2. AI analyzes your communication pattern.<br>
        3. Filler words and sentence structure are evaluated.<br>
        4. VocalIQ generates a communication score and
        improvement suggestions.
    </div>
    """)

    # --------------------------------------------------------
    # FOOTER
    # --------------------------------------------------------

    gr.HTML("""
    <div class="footer">
        VocalIQ • AI Communication Intelligence Platform
        <br>
        Educational demonstration project
    </div>
    """)

    # ========================================================
    # EVENTS
    # ========================================================

    text_analyze.click(
        fn=analyze_text_input,
        inputs=text_input,
        outputs=[
            score,
            words,
            sentences,
            fillers,
            communication_style,
            feedback
        ]
    )

    voice_analyze.click(
        fn=analyze_voice_input,
        inputs=audio_input,
        outputs=[
            transcript,
            score,
            words,
            fillers,
            communication_style,
            feedback
        ]
    )

    sample_btn.click(
        fn=load_sample,
        outputs=text_input
    )


# ============================================================
# LAUNCH
# ============================================================

demo.launch(
    share=True,
    debug=True
)

/tmp/ipykernel_1254/535873984.py:160: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://35f7bf5971a970a562.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
